# LC-MS/MS Data Explorer
Step through: **TIC → MS1 → MS2 → export `.ms`**

All plots are interactive — zoom, pan, hover for values.

In [ ]:
from pathlib import Path
import numpy as np
from pyteomics import mzml

MZML = Path("/home/magled/msms-structure-elucidation/data/QE_SN01781L_150430-04.mzML")
EXPORT_DIR = Path("/home/magled/msms-structure-elucidation/data/selected/")

ms1_scans, ms2_scans = [], []
with mzml.MzML(str(MZML)) as r:
    for s in r:
        rt = (
            float(s.get("scanList", {}).get("scan", [{}])[0].get("scan start time", 0))
            * 60
        )
        if s.get("ms level") == 1:
            ms1_scans.append(
                {
                    "rt": rt,
                    "mz": s.get("m/z array", np.array([])),
                    "intensity": s.get("intensity array", np.array([])),
                }
            )
        elif s.get("ms level") == 2:
            prec = s.get("precursorList", {}).get("precursor", [{}])[0]
            sel = prec.get("selectedIonList", {}).get("selectedIon", [{}])[0]
            act = prec.get("activation", {})
            ms2_scans.append(
                {
                    "rt": rt,
                    "precursor_mz": sel.get("selected ion m/z", 0),
                    "precursor_intensity": sel.get("peak intensity", 0),
                    "ce": act.get("collision energy", ""),
                    "mz": s.get("m/z array", np.array([])),
                    "intensity": s.get("intensity array", np.array([])),
                }
            )

rts = np.array([s["rt"] for s in ms1_scans])
tics = np.array([s["intensity"].sum() for s in ms1_scans])
print(f"Loaded {len(ms1_scans)} MS1 scans and {len(ms2_scans)} MS2 scans")

## 1. Total Ion Chromatogram (TIC)
Each peak = a compound eluting off the LC column. Hover to read the RT, zoom in to separate overlapping peaks.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=rts,
        y=tics,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        hovertemplate="RT: %{x:.1f}s<br>Intensity: %{y:.2e}<extra></extra>",
    )
)
fig.update_layout(
    title="TIC — note an RT of interest, then set RT_S in the next cell",
    xaxis_title="Retention time (s)",
    yaxis_title="Total intensity",
    height=300,
    margin=dict(t=40, b=40),
)
fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=rts / 60,
        y=tics,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        hovertemplate="RT: %{x:.1f}min<br>Intensity: %{y:.2e}<extra></extra>",
    )
)
fig.update_layout(
    title="TIC — note an RT of interest, then set RT_S in the next cell",
    xaxis_title="Retention time (min)",
    yaxis_title="Total intensity",
    height=300,
    margin=dict(t=40, b=40),
)
fig.show()

## 2. MS1 at a chosen retention time
Set `RT_S` to a retention time (seconds) from the TIC above.

In [ ]:
RT_S = 6.52 * 60  # <-- change this

scan = ms1_scans[int(np.argmin(np.abs(rts - RT_S)))]
print(f"Nearest MS1 scan: RT={scan['rt']:.1f}s")

# Build stick spectrum: for each peak add base→tip→base triplet (plotly trick for sticks)
mzs, ints = scan["mz"], scan["intensity"]
x = np.empty(len(mzs) * 3)
y = np.empty(len(mzs) * 3)
x[0::3] = mzs
x[1::3] = mzs
x[2::3] = np.nan
y[0::3] = 0
y[1::3] = ints
y[2::3] = np.nan

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        hovertemplate="m/z: %{x:.4f}<br>Intensity: %{y:.2e}<extra></extra>",
    )
)
fig.update_layout(
    title=f"MS1 at RT={scan['rt']:.1f}s — hover for m/z, zoom in, then set PRECURSOR_MZ below",
    xaxis_title="m/z",
    yaxis_title="Intensity",
    height=350,
    margin=dict(t=40, b=40),
)
fig.show()

top = sorted(zip(mzs, ints), key=lambda x: -x[1])[:10]
print(f"\nTop 10 peaks at RT={scan['rt']:.1f}s:")
print(f"{'m/z':>12}  {'intensity':>16}")
for m, i in top:
    print(f"{m:>12.4f}  {i:>16.0f}")

## 3. MS2 for a chosen precursor
Set `PRECURSOR_MZ` to one of the m/z values from the MS1 above.

In [ ]:
PRECURSOR_MZ = 397.1478  # <-- change this
TOL = 0.01

hits = [s for s in ms2_scans if abs(s["precursor_mz"] - PRECURSOR_MZ) <= TOL]
print(f"{len(hits)} MS2 scans for precursor m/z={PRECURSOR_MZ} ±{TOL}")

BIN_WIDTH = 0.01  # m/z bin width for averaging across scans

if hits:
    # average fragment spectrum across all matching scans, binned by m/z
    all_mz = np.concatenate([s["mz"] for s in hits])
    all_int = np.concatenate([s["intensity"] for s in hits])
    ce_vals = sorted({s["ce"] for s in hits})

    bin_idx = np.round(all_mz / BIN_WIDTH).astype(int)
    order = np.argsort(bin_idx)
    bin_idx, all_mz, all_int = bin_idx[order], all_mz[order], all_int[order]
    uniq_bins, start = np.unique(bin_idx, return_index=True)
    sum_int = np.add.reduceat(all_int, start)
    sum_mz = np.add.reduceat(all_mz * all_int, start)
    avg_mz = np.where(sum_int > 0, sum_mz / sum_int, uniq_bins * BIN_WIDTH)
    avg_int = sum_int / len(hits)

    avg = {"mz": avg_mz, "intensity": avg_int, "rt": np.mean([s["rt"] for s in hits])}

    x = np.empty(len(avg["mz"]) * 3)
    y = np.empty(len(avg["mz"]) * 3)
    x[0::3] = avg["mz"]
    x[1::3] = avg["mz"]
    x[2::3] = np.nan
    y[0::3] = 0
    y[1::3] = avg["intensity"]
    y[2::3] = np.nan

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            line=dict(width=1, color="darkorange"),
            hovertemplate="m/z: %{x:.4f}<br>Intensity: %{y:.2e}<extra></extra>",
        )
    )
    fig.update_layout(
        title=f"MS2  precursor={PRECURSOR_MZ}  CE={ce_vals}  (avg of {len(hits)} scans)",
        xaxis_title="m/z",
        yaxis_title="Intensity",
        height=350,
        margin=dict(t=40, b=40),
    )
    fig.show()
else:
    print("No MS2 found — try adjusting PRECURSOR_MZ or TOL")

## 4. Export as `.ms`
Saves the MS2 from the cell above to `EXPORT_DIR` for FRIGID/ICEBERG.

In [ ]:
if not hits:
    print("No hits — run cell 3 first.")
else:
    # averaged spectrum from cell 3 — what FRIGID/ICEBERG expect
    ce = ce_vals[0] if len(ce_vals) == 1 else "/".join(str(c) for c in ce_vals)
    label = f"collision {ce} eV" if ce else "collision"

    lines = [
        f">compound {MZML.stem}_mz{round(PRECURSOR_MZ, 4)}",
        f">parentmass {round(PRECURSOR_MZ, 4)}",
        ">ionization [M+H]+",
        f">rt {round(avg['rt'], 1)}s",
        "",
        f">{label}",
    ]
    for mz, inten in sorted(zip(avg["mz"].tolist(), avg["intensity"].tolist())):
        lines.append(f"{mz}\t{inten}")

    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    out = EXPORT_DIR / f"mz{round(PRECURSOR_MZ, 4)}.ms"
    out.write_text("\n".join(lines) + "\n")
    print(
        f"Exported avg spectrum (RT={avg['rt']:.1f}s, CE={ce}, {len(hits)} scans): {out}"
    )

## 5. Automated features (`msms-feature-detect` / MZmine output)

The cells above are the **manual** picking path (pick an RT, pick a precursor, guess a
tolerance). `msms-feature-detect` automates all of that: MZmine detects every
chromatographic feature in the file, pairs each with its MS2 scans (by real isolation
window + retention-time overlap), averages the fragments, and exports one spectrum per
feature to a SIRIUS `.mgf` file.

Set `MGF_PATH` to that file to browse the automated results — a feature table (m/z, RT,
height) plus a spectrum viewer, no manual picking required.

In [ ]:
MGF_PATH = Path(
    "/home/magled/msms-structure-elucidation/.agents/test/feature-detect/QE_SN01781L_150430-04_sirius.mgf"
)


def parse_sirius_mgf(path):
    """Parse a SIRIUS-format .mgf into a list of MS2 feature dicts.

    Each feature has two IONS blocks (an MS1 snapshot at the apex, then the
    averaged MS2 fragment spectrum tagged MSLEVEL=2) — only the MS2 block is kept.
    """
    features = []
    meta, mzs, ints = {}, [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line == "BEGIN IONS":
                meta, mzs, ints = {}, [], []
            elif line == "END IONS":
                if meta.get("MSLEVEL") == "2":
                    features.append(
                        {**meta, "mz": np.array(mzs), "intensity": np.array(ints)}
                    )
            elif "=" in line and " " not in line.split("=", 1)[1][:1]:
                key, _, val = line.partition("=")
                meta[key] = val
            elif line and line[0].isdigit():
                m, i = line.split()
                mzs.append(float(m))
                ints.append(float(i))
    return features


features = parse_sirius_mgf(MGF_PATH)
print(f"Loaded {len(features)} MS2 features from {MGF_PATH.name}")

print(
    f"\n{'FEATURE_ID':>10}  {'PEPMASS':>10}  {'RT (s)':>8}  {'MS1 height':>12}  {'ionmode':>8}  {'n_peaks':>8}"
)
for ft in features:
    print(
        f"{ft.get('FEATURE_ID', ''):>10}  {float(ft.get('PEPMASS', 0)):>10.4f}  "
        f"{float(ft.get('RTINSECONDS', 0)):>8.1f}  {float(ft.get('FEATURE_MS1_HEIGHT', 0)):>12.2e}  "
        f"{ft.get('IONMODE', ''):>8}  {len(ft['mz']):>8}"
    )

Set `FEATURE_INDEX` to a row from the table above (0-based) to plot its averaged MS2 spectrum.

In [ ]:
FEATURE_INDEX = 0  # <-- change this

ft = features[FEATURE_INDEX]
mzs, ints = ft["mz"], ft["intensity"]

x = np.empty(len(mzs) * 3)
y = np.empty(len(mzs) * 3)
x[0::3] = mzs
x[1::3] = mzs
x[2::3] = np.nan
y[0::3] = 0
y[1::3] = ints
y[2::3] = np.nan

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        line=dict(width=1, color="seagreen"),
        hovertemplate="m/z: %{x:.4f}<br>Intensity: %{y:.2e}<extra></extra>",
    )
)
fig.update_layout(
    title=(
        f"Feature {ft.get('FEATURE_ID')}  precursor={float(ft.get('PEPMASS', 0)):.4f}  "
        f"RT={float(ft.get('RTINSECONDS', 0)):.1f}s  ionmode={ft.get('IONMODE')}  "
        f"({ft.get('SPECTYPE')}, {len(mzs)} peaks)"
    ),
    xaxis_title="m/z",
    yaxis_title="Intensity",
    height=350,
    margin=dict(t=40, b=40),
)
fig.show()

### Feature map
All detected features at once — RT vs precursor m/z, marker size/color by MS1 height.
Something the one-precursor-at-a-time manual view above can't show.

In [ ]:
feat_rt = np.array([float(ft.get("RTINSECONDS", 0)) for ft in features])
feat_mz = np.array([float(ft.get("PEPMASS", 0)) for ft in features])
feat_height = np.array([float(ft.get("FEATURE_MS1_HEIGHT", 0)) for ft in features])
feat_ionmode = [ft.get("IONMODE", "") for ft in features]
feat_id = [ft.get("FEATURE_ID", "") for ft in features]

fig = go.Figure()
for mode, color in [("POSITIVE", "steelblue"), ("NEGATIVE", "firebrick")]:
    mask = [m == mode for m in feat_ionmode]
    if not any(mask):
        continue
    fig.add_trace(
        go.Scatter(
            x=feat_rt[mask],
            y=feat_mz[mask],
            mode="markers",
            name=mode,
            marker=dict(
                size=8 + 12 * (feat_height[mask] / feat_height.max()),
                color=color,
                opacity=0.7,
            ),
            text=[f"feature {i}" for i in np.array(feat_id)[mask]],
            hovertemplate="RT: %{x:.1f}s<br>m/z: %{y:.4f}<br>%{text}<extra></extra>",
        )
    )
fig.update_layout(
    title=f"Feature map — {len(features)} features (color = ion mode, size = MS1 height)",
    xaxis_title="Retention time (s)",
    yaxis_title="Precursor m/z",
    height=450,
    margin=dict(t=40, b=40),
)
fig.show()